In [ ]:
import pandas as pd
from glob import glob
import numpy as np
import sys
sys.path.append("..")
from methods.czi_preprocessing import CZIPreprocessing

In [ ]:
def get_max_size(npz_list, n_dim=3):
    """
    Get the maximum size (shape) among a list of Numpy arrays.

    This function iterates through a list of Numpy arrays stored in .npz files,
    extracts their shapes, and returns the maximum size among all arrays.

    Args:
        npz_list (list of str): A list of file paths to .npz files containing Numpy arrays.
        n_dim (int): The number of dimensions expected in the arrays (default is 3).

    Returns:
        tuple of int: A tuple representing the maximum size of the arrays.
    """
    # Initialize an empty array to store sizes
    array_sizes = np.empty(shape=(len(npz_list), n_dim))

    # Iterate through the .npz files
    for i, nuc_path in enumerate(npz_list):
        img = np.load(nuc_path)["img"]
        array_sizes[i] = img.shape

    # Find the maximum size among all arrays
    max_size = array_sizes.max(axis=0)

    # Convert the resulting tuple of floats to integers
    max_size = tuple(int(x) for x in max_size)

    return max_size

In [ ]:
# Set the resolution for image preprocessing (in micrometers)
res = 0.05

# Choose a normalization method for image processing ("none" for no normalization)
norm = "none"

# Set a random seed for reproducibility
seed = 54321

# Define a list of conditions for image processing
conditions = [
    "aged",
    "aged_CASIN",
    "aged_DMSO",
    "aged_IOX",
    "aged_UNC",
    "aged_treated_RhoAi",
    "young",
]

# Load a list of image paths identified as outliers
outliers = pd.read_csv("../results/czi_image_outliers.csv")["path"].values

# Initialize an instance of the CZIPreprocessing class
czi_prep = CZIPreprocessing(
    input_dir="../../data/raw_data",
    output_dir=f"../../data/preprocessed/3D_res={res}_norm={norm}_final_npz",
    conditions=conditions,
    resolution=res,
    normalization=norm,
    channels=["Ch1", "DAPI"],  # Channels to be used from CZI images
    resize=True,
    outliers=outliers,
    override=False,  # Save mode for already processed images
)

In [ ]:
import time
start_time = time.time()

# Preprocess and save as 3D arrays in .npz files
output_paths = czi_prep.save_as_3D_array_multiprocessing(n_cores=20)

end_time = time.time()
print(f"Total time: {end_time - start_time:.2f} seconds")

In [ ]:
# Merge both young and aged filenames in the same list
aged_path_list = glob(f"{czi_prep.output_dir}/aged/*.npz")
young_path_list = glob(f"{czi_prep.output_dir}/young/*.npz")
all_list = young_path_list + aged_path_list

# Find the maximum size for each dimension
max_size = get_max_size(all_list, n_dim=3)
print(max_size)

In [ ]:
if res == 0.2:
    max_size = (72, 72)
elif res == 0.1:
    max_size = (128, 128)
elif res == 0.05:
    max_size = (224, 224)  # Standard for Mobilenet

In [ ]:
# Define a list of conditions for processing (e.g., "young" and "aged")
conditions = ["young", "aged"]

# Define class numbers to be used for labeling (e.g., 0 for "young" and 1 for "aged")
class_nums = (1, 0)

# Define the plane in the 3 Dimensions to perform the slicing
slide_plane = "XY"

norm="minmax"

# Define the output directory for preprocessed 2D images with specified parameters
output_img_dir = f"../../data/preprocessed/2D_res={res}_norm={norm}_{slide_plane}_final"

# Perform conversion of NPZ files to Keras-compatible PNG images
czi_prep.npz_to_keras_png(
    output_img_dir,
    conditions=conditions,
    channel_mode="L",  # Color channel mode (L for grayscale)
    special_mode=None,
    slide_plane=slide_plane,
    class_nums=class_nums,
    datasets=["train", "validation"],  # List of datasets
    data_splits=[0.8, 0.2],  # Data split ratios for datasets
    min_nuc_ratio=0.5,  # Minimum nucleus mask ratio
    standardize=True,
    fix_size=max_size,  # Maximum image size
    seed=seed,  # Random seed for reproducibility
    save_mask=False,
)

In [ ]:
# IN ORDER TO ADD AGED DMSO TO THE AGED DATASET
conditions = ["aged_DMSO"]

# Define class numbers to be used for labeling (e.g., 0 for "young" and 1 for "aged")
class_nums = (0,)

# Define the plane in the 3 Dimensions to perform the slicing
slide_plane = "XY"

norm="none"

# Define the output directory for preprocessed 2D images with specified parameters
output_img_dir = f"../../data/preprocessed/2D_res={res}_norm={norm}_{slide_plane}_final_withDMSO"

# Perform conversion of NPZ files to Keras-compatible PNG images
czi_prep.npz_to_keras_png(
    output_img_dir,
    conditions=conditions,
    channel_mode="L",  # Color channel mode (L for grayscale)
    special_mode=None,
    slide_plane=slide_plane,
    class_nums=class_nums,
    datasets=["train", "validation"],  # List of datasets
    data_splits=[0.8, 0.2],  # Data split ratios for datasets
    min_nuc_ratio=0.5,  # Minimum nucleus mask ratio
    standardize=False,
    fix_size=max_size,  # Maximum image size
    seed=seed,  # Random seed for reproducibility
    save_mask=False,
)

In [ ]:
k_cv = 5
norm = "none"
output_img_dir = f"../../data/preprocessed/2D_res={res}_norm={norm}_{k_cv}fold_withDMSO"
conditions = ["young", "aged", "aged_DMSO"]
class_nums = (1, 0, 0)

test_prop = 0.0
prop = 1.0 / k_cv
data_split = [prop] * k_cv
datasets = [f"fold_{k}" for k in range(1, k_cv + 1)]

czi_prep.npz_to_keras_png(
    output_img_dir,
    conditions=conditions,
    channel_mode="L",
    special_mode=None,
    slide_plane="XY",
    class_nums=class_nums,
    datasets=datasets,
    data_splits=data_split,
    min_nuc_ratio=0.5,
    standardize=False,
    fix_size=max_size,
    seed=seed,
    save_mask=False,
)

***